In [5]:
import os
import re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import statsmodels.formula.api as smf

import warnings

os.chdir('../../..')

warnings.filterwarnings("ignore")

# --- helpers ---

def load_band_table(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)

    # numeric ids
    for c in ("s_id", "t_id", "age", "label", "stim_label"):
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # string-like
    for c in ("gender", "handiness", "day_time", "stim_type", "task_type"):
        if c in df.columns:
            df[c] = df[c].astype("string")

    return df


def infer_feature_cols(df: pd.DataFrame):
    """Возвращает список колонок вида ch{N}_{Band}"""
    cols = []
    for c in df.columns:
        if isinstance(c, str) and re.match(r"^ch\d+_.+$", c):
            cols.append(c)
    return cols


def parse_ch_band(col: str):
    m = re.match(r"^ch(\d+)_(.+)$", col)
    if not m:
        return None, None
    return int(m.group(1)), m.group(2)


def map_to_group_series(df: pd.DataFrame, feature_name: str) -> pd.Series:
    """
    Превращает сырые значения фичи в групповые метки из FEATURE_GROUPS[feature].
    Возвращает Series с метками групп (строки) или <NA>.
    """
    groups = FEATURE_GROUPS[feature_name]  # {group_label: [allowed_values...]}

    # где брать исходное значение
    src_col = feature_name
    if src_col not in df.columns:
        # совместимость: иногда stim_label лежит в label
        if feature_name == "stim_label" and "label" in df.columns:
            src_col = "label"
        else:
            raise KeyError(f"Нет колонки '{feature_name}' в band_table")

    s = df[src_col]

    out = pd.Series(pd.NA, index=df.index, dtype="string")

    for g_label, allowed in groups.items():
        allowed_set = set(allowed)

        if feature_name in ("age", "stim_label"):
            # числовые
            mask = s.isin(list(allowed_set))
        else:
            # строковые
            mask = s.astype("string").isin([str(x) for x in allowed_set])

        out.loc[mask] = str(g_label)

    return out


In [7]:
def run_interaction_for_pair(
    band_table_path: str,
    feature_A: str,
    feature_B: str,
    alpha: float = 0.05,
    min_subjects: int = 8,
    save_dir: str = "./Supplementary",
):
    df = load_band_table(band_table_path)

    # только строки с валидным s_id
    df = df[df["s_id"].notna()].copy()

    # маппинг в групповые метки (по FEATURE_GROUPS)
    A = map_to_group_series(df, feature_A)
    B = map_to_group_series(df, feature_B)

    df["_A"] = A
    df["_B"] = B

    # выбрасываем строки не попавшие в группы
    df = df[df["_A"].notna() & df["_B"].notna()].copy()

    # минимум по числу людей
    n_subj = df["s_id"].nunique()
    if n_subj < min_subjects:
        raise RuntimeError(f"Слишком мало субъектов для модели: {n_subj} < {min_subjects}")

    feat_cols = infer_feature_cols(df)
    rows = []

    pbar = tqdm(feat_cols, desc=f"Interaction: {feature_A}×{feature_B}", unit="feat", dynamic_ncols=True)
    for col in pbar:
        ch, band = parse_ch_band(col)
        if ch is None:
            continue

        d = df[["s_id", "_A", "_B", col]].copy()
        d[col] = pd.to_numeric(d[col], errors="coerce")
        d = d[d[col].notna()].copy()
        if d.empty:
            continue

        # требования к уникальным уровням
        if d["_A"].nunique() < 2 or d["_B"].nunique() < 2:
            continue

        # минимум субъектов в данных именно этого признака
        if d["s_id"].nunique() < min_subjects:
            continue

        try:
            # full model
            model = smf.ols(f"{col} ~ C(_A) * C(_B)", data=d)
            res = model.fit(cov_type="cluster", cov_kwds={"groups": d["s_id"]})
        except Exception:
            continue

        # выбираем параметры interaction
        param_names = list(res.params.index)
        inter_idx = [
            i for i, name in enumerate(param_names)
            if (":" in name) and ("C(_A)" in name) and ("C(_B)" in name)
        ]
        if not inter_idx:
            continue

        # joint Wald test: все interaction = 0
        R = np.zeros((len(inter_idx), len(param_names)), dtype=float)
        for r, j in enumerate(inter_idx):
            R[r, j] = 1.0

        try:
            wt = res.wald_test(R)  # robust cov already inside res
            p_int = float(wt.pvalue)
            stat = float(np.asarray(wt.statistic).reshape(-1)[0])
            df_int = int(getattr(wt, "df_denom", getattr(wt, "df_num", len(inter_idx))))
        except Exception:
            p_int = np.nan
            stat = np.nan
            df_int = len(inter_idx)

        # удобно: коэффициент interaction (если 2x2, он один)
        if len(inter_idx) == 1:
            inter_name = param_names[inter_idx[0]]
            inter_coef = float(res.params.iloc[inter_idx[0]])
            inter_p = float(res.pvalues.iloc[inter_idx[0]])
        else:
            inter_name = "MULTI"
            inter_coef = np.nan
            inter_p = np.nan

        rows.append({
            "feature_A": feature_A,
            "feature_B": feature_B,
            "channel": ch,
            "band": band,
            "n_obs": int(len(d)),
            "n_subjects": int(d["s_id"].nunique()),
            "levels_A": int(d["_A"].nunique()),
            "levels_B": int(d["_B"].nunique()),
            "p_interaction_joint": p_int,
            "stat_interaction_joint": stat,
            "df_interaction_terms": len(inter_idx),
            "interaction_term": inter_name,
            "interaction_coef": inter_coef,
            "interaction_p_single": inter_p,  # только если 2x2
        })

        pbar.set_postfix({"p_int": f"{p_int:.3g}" if np.isfinite(p_int) else "nan"})

    out = pd.DataFrame(rows)
    if not out.empty:
        out = out.sort_values(["p_interaction_joint", "n_subjects"], ascending=[True, False]).reset_index(drop=True)

    # save
    os.makedirs(save_dir, exist_ok=True)
    safeA = feature_A.replace("/", "-")
    safeB = feature_B.replace("/", "-")
    out_path = os.path.join(save_dir, f"interaction__{safeA}x{safeB}.csv")
    out.to_csv(out_path, index=False)

    return out


In [10]:
band_table_path = "./Supplementary/Analysis/band_table.csv"

pairs = [
    ("day_time", "gender"),
    ("stim_type", "gender"),
    ("stim_label", "gender"),
    ("day_time", "age"),
    ("day_time", "handiness"),
    ("age", "handiness"),
]
FEATURE_GROUPS = {
    "day_time": {
        "Day": ["Day"],
        "Evening": ["Evening"],
    },
    "stim_type": {
        "r": ["r"],
        "g": ["g"],
    },
    "stim_label": {
        "line":   [0, 1, 4, 6, 7, 11, 8],
        "figure": [2, 3, 5, 9, 10, 12],
    },
    "gender": {
        "m": ["m"],
        "f": ["f"],
    },
    "age": {
        "18-22": [18, 19, 20, 21, 22],
        "23-29": [23, 24, 25, 25, 26, 27, 28, 29],
        "30-35": [30, 31, 32, 33, 34, 35],
    },
    "handiness": {
        "l": ["l"],
        "r": ["r"],
    },
}
interaction_results = {}
for a, b in pairs:
    df_int = run_interaction_for_pair(
        band_table_path=band_table_path,
        feature_A=a,
        feature_B=b,
        alpha=0.05,
        min_subjects=8,
        save_dir="Data/EEG-Visual-Experiment/Supplementary/Analysis",
    )
    interaction_results[(a, b)] = df_int
    print(f"\nTOP interaction: {a} × {b}")
    display(df_int.head(15))


Interaction: day_time×gender:   0%|          | 0/252 [00:00<?, ?feat/s]


TOP interaction: day_time × gender


,feature_A,feature_B,channel,band,n_obs,n_subjects,levels_A,levels_B,p_interaction_joint,stat_interaction_joint,df_interaction_terms,interaction_term,interaction_coef,interaction_p_single
0,day_time,gender,28,Beta,945,24,2,2,0.000021,18.122474,1,C(_A)[T.Evening]:C(_B)[T.m],0.095280,0.000021
1,day_time,gender,6,Beta,945,24,2,2,0.004971,7.890119,1,C(_A)[T.Evening]:C(_B)[T.m],0.034150,0.004971
2,day_time,gender,33,Beta,945,24,2,2,0.006831,7.316954,1,C(_A)[T.Evening]:C(_B)[T.m],0.074790,0.006831
3,day_time,gender,31,Beta,945,24,2,2,0.007185,7.226123,1,C(_A)[T.Evening]:C(_B)[T.m],0.289561,0.007185
4,day_time,gender,23,Beta,945,24,2,2,0.008290,6.969776,1,C(_A)[T.Evening]:C(_B)[T.m],0.037626,0.008290
5,day_time,gender,32,Beta,945,24,2,2,0.022981,5.169963,1,C(_A)[T.Evening]:C(_B)[T.m],0.169431,0.022981
6,day_time,gender,21,Beta,945,24,2,2,0.026856,4.900073,1,C(_A)[T.Evening]:C(_B)[T.m],0.025916,0.026856
7,day_time,gender,60,Beta,945,24,2,2,0.031369,4.632630,1,C(_A)[T.Evening]:C(_B)[T.m],0.144568,0.031369
8,day_time,gender,12,Tetta,945,24,2,2,0.037335,4.335069,1,C(_A)[T.Evening]:C(_B)[T.m],-0.112482,0.037335
9,day_time,gender,5,Beta,945,24,2,2,0.040900,4.180164,1,C(_A)[T.Evening]:C(_B)[T.m],0.054953,0.040900


Interaction: stim_type×gender:   0%|          | 0/252 [00:00<?, ?feat/s]


TOP interaction: stim_type × gender


...


TOP interaction: day_time × handiness


,feature_A,feature_B,channel,band,n_obs,n_subjects,levels_A,levels_B,p_interaction_joint,stat_interaction_joint,df_interaction_terms,interaction_term,interaction_coef,interaction_p_single
0,day_time,handiness,8,Tetta,924,24,2,2,3.025316e-131,594.221078,1,C(_A)[T.Evening]:C(_B)[T.r],-0.889194,3.025316e-131
1,day_time,handiness,12,Beta,924,24,2,2,4.695550e-119,538.178549,1,C(_A)[T.Evening]:C(_B)[T.r],-0.277502,4.695550e-119
2,day_time,handiness,61,Beta,924,24,2,2,1.817602e-86,388.430786,1,C(_A)[T.Evening]:C(_B)[T.r],-0.322512,1.817602e-86
3,day_time,handiness,8,Delta,924,24,2,2,2.555466e-66,295.916401,1,C(_A)[T.Evening]:C(_B)[T.r],-1.043237,2.555466e-66
4,day_time,handiness,8,Alpha,924,24,2,2,4.455874e-62,276.451317,1,C(_A)[T.Evening]:C(_B)[T.r],-0.789280,4.455874e-62
5,day_time,handiness,40,Tetta,924,24,2,2,1.842776e-43,191.085137,1,C(_A)[T.Evening]:C(_B)[T.r],-0.439366,1.842776e-43
6,day_time,handiness,8,Beta,924,24,2,2,1.087132e-31,137.205804,1,C(_A)[T.Evening]:C(_B)[T.r],-0.564451,1.087132e-31
7,day_time,handiness,61,Tetta,924,24,2,2,1.601613e-22,95.342212,1,C(_A)[T.Evening]:C(_B)[T.r],-0.323132,1.601613e-22
8,day_time,handiness,40,Alpha,924,24,2,2,2.031457e-20,85.760084,1,C(_A)[T.Evening]:C(_B)[T.r],-0.383044,2.031457e-20
9,day_time,handiness,40,Delta,924,24,2,2,1.771409e-15,63.304037,1,C(_A)[T.Evening]:C(_B)[T.r],-0.470687,1.771409e-15


Interaction: age×handiness:   0%|          | 0/252 [00:00<?, ?feat/s]


TOP interaction: age × handiness


,feature_A,feature_B,channel,band,n_obs,n_subjects,levels_A,levels_B,p_interaction_joint,stat_interaction_joint,df_interaction_terms,interaction_term,interaction_coef,interaction_p_single
0,age,handiness,57,Beta,1197,31,3,2,0.966069,0.069040,2,MULTI,NaN,NaN
1,age,handiness,21,Beta,1197,31,3,2,0.976646,0.047263,2,MULTI,NaN,NaN
2,age,handiness,48,Beta,1197,31,3,2,0.977502,0.045510,2,MULTI,NaN,NaN
3,age,handiness,60,Delta,1197,31,3,2,0.978413,0.043647,2,MULTI,NaN,NaN
4,age,handiness,17,Beta,1197,31,3,2,0.979010,0.042427,2,MULTI,NaN,NaN
5,age,handiness,32,Delta,1197,31,3,2,0.979703,0.041013,2,MULTI,NaN,NaN
6,age,handiness,0,Alpha,1197,31,3,2,0.980425,0.039537,2,MULTI,NaN,NaN
7,age,handiness,33,Beta,1197,31,3,2,0.980673,0.039033,2,MULTI,NaN,NaN
8,age,handiness,38,Beta,1197,31,3,2,0.981327,0.037700,2,MULTI,NaN,NaN
9,age,handiness,59,Beta,1197,31,3,2,0.982678,0.034947,2,MULTI,NaN,NaN
